In [58]:
import pandas as pd
import numpy as np

print("Evaluation notebook ready!")

Evaluation notebook ready!


In [59]:
# Load KuaiRec data

small_matrix = pd.read_csv("../data/small_matrix.csv")
item_features = pd.read_csv("../data/item_daily_features.csv")

# Create capped watch score
small_matrix["watch_score"] = small_matrix["watch_ratio"].clip(upper=3.0)

# Remove interactions without a valid date
small_matrix = small_matrix.dropna(subset=["date"]).copy()

# Convert YYYYMMDD date format
small_matrix["date"] = pd.to_datetime(
    small_matrix["date"].astype(int).astype(str),
    format="%Y%m%d"
)

# Temporal split
train_data = small_matrix[
    small_matrix["date"] <= pd.Timestamp("2020-08-29")
].copy()

test_data = small_matrix[
    small_matrix["date"] > pd.Timestamp("2020-08-29")
].copy()

print("Training interactions:", len(train_data))
print("Test interactions:", len(test_data))
print("Training users:", train_data["user_id"].nunique())
print("Training videos:", train_data["video_id"].nunique())
print("Test users:", test_data["user_id"].nunique())
print("Test videos:", test_data["video_id"].nunique())

Training interactions: 4387382
Test interactions: 107196
Training users: 1411
Training videos: 3288
Test users: 1411
Test videos: 1162


In [60]:
# Inspect the structure of the caption/category file

video_tags_raw = pd.read_csv(
    "../data/kuairec_caption_category.csv",
    engine="python",
    nrows=5
)

print("Columns:")
print(video_tags_raw.columns.tolist())

print("\nFirst 5 rows:")
display(video_tags_raw)

Columns:
['video_id', 'manual_cover_text', 'caption', 'topic_tag', 'first_level_category_id', 'first_level_category_name', 'second_level_category_id', 'second_level_category_name', 'third_level_category_id', 'third_level_category_name']

First 5 rows:


,video_id,manual_cover_text,caption,topic_tag,first_level_category_id,first_level_category_name,second_level_category_id,second_level_category_name,third_level_category_id,third_level_category_name
0,0,UNKNOWN,精神小伙路难走 程哥你狗粮慢点撒,[],8,颜值,673,颜值随拍,-124,UNKNOWN
1,1,UNKNOWN,NaN,[],27,高新数码,-124,UNKNOWN,-124,UNKNOWN
2,2,UNKNOWN,晚饭后，运动一下！,[],9,喜剧,727,搞笑互动,-124,UNKNOWN
3,3,UNKNOWN,我平淡无奇，惊艳不了时光，温柔不了岁月，我只想漫无目的的走走，努力发笔小财，给自己买花 自己长大.,[],26,摄影,686,主题摄影,2434,景物摄影
4,4,五爱街最美美女 一天1q,#搞笑 #感谢快手我要上热门 #五爱市场 这真是完美搭配啊！,"[五爱市场,感谢快手我要上热门,搞笑]",5,时尚,737,营销售卖,2596,女装


In [61]:
# Build video-to-tag mapping

video_tags = pd.read_csv(
    "../data/kuairec_caption_category.csv",
    engine="python"
)

# Keep videos present in the training interaction data
video_tags = video_tags[
    video_tags["video_id"].isin(train_data["video_id"].unique())
].copy()

# Rename topic_tag to the name used throughout our project
video_tags = video_tags.rename(
    columns={"topic_tag": "video_tag"}
)

# Keep one row per video-tag pair
video_tags = video_tags.drop_duplicates(
    subset=["video_id", "video_tag"]
)

print("Videos with tags:", video_tags["video_id"].nunique())
print("Unique tags:", video_tags["video_tag"].nunique())
print("Video-tag rows:", len(video_tags))

Videos with tags: 0
Unique tags: 0
Video-tag rows: 0


In [62]:
print("Training video_id dtype:", train_data["video_id"].dtype)
print("Tag file video_id dtype:", video_tags["video_id"].dtype)

print("\nTraining video IDs:")
print(train_data["video_id"].head().tolist())

print("\nTag file video IDs:")
print(video_tags["video_id"].head().tolist())

Training video_id dtype: int64
Tag file video_id dtype: object

Training video IDs:
[148, 183, 3649, 5262, 8234]

Tag file video IDs:
[]


In [63]:
# Reload the tag file from scratch

video_tags = pd.read_csv(
    "../data/kuairec_caption_category.csv",
    engine="python"
)

# Normalize video_id types
video_tags["video_id"] = pd.to_numeric(
    video_tags["video_id"],
    errors="coerce"
)

video_tags["video_id"] = video_tags["video_id"].astype("Int64")

train_video_ids = pd.to_numeric(
    train_data["video_id"],
    errors="coerce"
).astype("Int64")

print("Training video IDs:", train_video_ids.nunique())
print("Tag file video IDs:", video_tags["video_id"].nunique())

print("\nExample training IDs:")
print(train_video_ids.head().tolist())

print("\nExample tag IDs:")
print(video_tags["video_id"].head().tolist())

Training video IDs: 3288
Tag file video IDs: 10728

Example training IDs:
[148, 183, 3649, 5262, 8234]

Example tag IDs:
[0, 1, 2, 3, 4]


In [64]:
# Keep only videos that appear in the training interactions

video_tags = video_tags[
    video_tags["video_id"].isin(train_video_ids)
].copy()

print("Videos with metadata:", video_tags["video_id"].nunique())
print("Total metadata rows:", len(video_tags))

Videos with metadata: 3288
Total metadata rows: 3288


In [65]:
def parse_topic_tags(value):
    if pd.isna(value):
        return []
    
    value = str(value).strip()
    
    # Remove the outer brackets
    if value.startswith("[") and value.endswith("]"):
        value = value[1:-1]
    
    # Empty tag list
    if not value.strip():
        return []
    
    # Split individual tags
    tags = [
        tag.strip()
        for tag in value.split(",")
        if tag.strip()
    ]
    
    return tags


video_tags["parsed_tags"] = video_tags["topic_tag"].apply(parse_topic_tags)

print(
    "Videos with at least one topic tag:",
    (video_tags["parsed_tags"].str.len() > 0).sum()
)

unique_topic_tags = set(
    tag
    for tags in video_tags["parsed_tags"]
    for tag in tags
)

print("Unique topic tags:", len(unique_topic_tags))

print("\nExample parsed tags:")
display(
    video_tags[
        ["video_id", "topic_tag", "parsed_tags"]
    ].head()
)

Videos with at least one topic tag: 1803
Unique topic tags: 1921

Example parsed tags:


,video_id,topic_tag,parsed_tags
103,103,"[快手九周年,爱官方爱热门支持快手传播正能量]","[快手九周年, 爱官方爱热门支持快手传播正能量]"
109,109,"[ClMA骑士,中国摩博会]","[ClMA骑士, 中国摩博会]"
120,120,[],[]
122,122,"[搞笑,搞笑配音]","[搞笑, 搞笑配音]"
128,128,[],[]


In [66]:
print(
    "Videos with first-level category:",
    video_tags["first_level_category_name"]
        .replace("UNKNOWN", np.nan)
        .notna()
        .sum()
)

print(
    "Unique first-level categories:",
    video_tags["first_level_category_name"]
        .replace("UNKNOWN", np.nan)
        .nunique()
)

print("\nTop first-level categories:")
display(
    video_tags["first_level_category_name"]
        .replace("UNKNOWN", np.nan)
        .value_counts()
        .head(10)
)

Videos with first-level category: 3284
Unique first-level categories: 37

Top first-level categories:


first_level_category_name
民生资讯    634
颜值      291
明星娱乐    243
其他      181
摄影      139
舞蹈      135
运动      134
美食      128
时尚      123
生活      122
Name: count, dtype: int64

In [67]:
# Build video-to-category mapping

video_categories = video_tags[
    ["video_id", "first_level_category_name"]
].copy()

# Remove unknown categories
video_categories = video_categories[
    video_categories["first_level_category_name"] != "UNKNOWN"
].copy()

# Remove duplicate video-category pairs
video_categories = video_categories.drop_duplicates()

print("Videos with category:", video_categories["video_id"].nunique())
print(
    "Unique first-level categories:",
    video_categories["first_level_category_name"].nunique()
)

print("\nExample mapping:")
display(video_categories.head())

Videos with category: 3286
Unique first-level categories: 37

Example mapping:


,video_id,first_level_category_name
103,103,美食
109,109,汽车
120,120,喜剧
122,122,喜剧
128,128,美食


In [68]:
# Build initial user category profile from training data

train_category = train_data.merge(
    video_categories,
    on="video_id",
    how="inner"
)

initial_user_category = (
    train_category
    .groupby(
        ["user_id", "first_level_category_name"]
    )
    .agg(
        interactions=("video_id", "count"),
        avg_watch_score=("watch_score", "mean")
    )
    .reset_index()
)

# Smoothed category preference
global_mean = train_category["watch_score"].mean()
smoothing = 5

initial_user_category["category_score"] = (
    (
        initial_user_category["interactions"]
        * initial_user_category["avg_watch_score"]
        + smoothing * global_mean
    )
    /
    (
        initial_user_category["interactions"]
        + smoothing
    )
)

print(
    "User-category profile rows:",
    len(initial_user_category)
)

print(
    "Users:",
    initial_user_category["user_id"].nunique()
)

print(
    "Categories represented:",
    initial_user_category["first_level_category_name"].nunique()
)

print("\nExample:")
display(initial_user_category.head(10))

User-category profile rows: 52193
Users: 1411
Categories represented: 37

Example:


,user_id,first_level_category_name,interactions,avg_watch_score,category_score
0,14,三农,17,0.808800,0.821023
1,14,二次元,76,1.003247,0.994564
2,14,亲子,86,1.072189,1.060672
3,14,人文,3,0.808524,0.842309
4,14,健康,40,0.616781,0.644092
5,14,健身,43,1.125286,1.097921
6,14,其他,171,0.860585,0.860642
7,14,军事,32,0.965068,0.951218
8,14,喜剧,115,0.866789,0.866613
9,14,奇人异象,11,0.915659,0.899072


In [69]:
# Assign initial category interest levels

initial_user_category["interest_level"] = np.select(
    [
        initial_user_category["interactions"] >= 8,
        initial_user_category["interactions"] >= 1
    ],
    [
        "Established",
        "Weak"
    ],
    default="Unseen"
)

print(
    initial_user_category["interest_level"].value_counts()
)

print("\nUsers represented:",
      initial_user_category["user_id"].nunique())

print("\nWeak category profiles:",
      (initial_user_category["interest_level"] == "Weak").sum())

print("Established category profiles:",
      (initial_user_category["interest_level"] == "Established").sum())

interest_level
Established    47153
Weak            5040
Name: count, dtype: int64

Users represented: 1411

Weak category profiles: 5040
Established category profiles: 47153


In [70]:
# Build Round 1 exploration candidates from weak categories

weak_categories = initial_user_category[
    initial_user_category["interest_level"] == "Weak"
][
    ["user_id", "first_level_category_name", "category_score"]
].copy()

# Connect weak categories to videos
round1_candidates = weak_categories.merge(
    video_categories,
    on="first_level_category_name",
    how="inner"
)

# Find videos already watched by each user
watched_pairs = train_data[
    ["user_id", "video_id"]
].drop_duplicates()

# Remove previously watched videos
round1_candidates = round1_candidates.merge(
    watched_pairs.assign(watched=1),
    on=["user_id", "video_id"],
    how="left"
)

round1_candidates = round1_candidates[
    round1_candidates["watched"].isna()
].drop(columns=["watched"])

print("Round 1 exploration candidates:",
      len(round1_candidates))

print("Users:",
      round1_candidates["user_id"].nunique())

print("Unique videos:",
      round1_candidates["video_id"].nunique())

print("Unique weak categories:",
      
      round1_candidates["first_level_category_name"].nunique())

Round 1 exploration candidates: 2804
Users: 1103
Unique videos: 56
Unique weak categories: 8


In [71]:
# Inspect the item category file used for the detailed interest representation

item_categories = pd.read_csv(
    "../data/item_categories.csv"
)

print("Columns:")
print(item_categories.columns.tolist())

print("\nShape:")
print(item_categories.shape)

print("\nFirst 5 rows:")
display(item_categories.head())

Columns:
['video_id', 'feat']

Shape:
(10728, 2)

First 5 rows:


,video_id,feat
0,0,[8]
1,1,"[27, 9]"
2,2,[9]
3,3,[26]
4,4,[5]


In [72]:
import ast

def parse_feat(value):
    if pd.isna(value):
        return []
    
    value = str(value).strip()
    
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return parsed
    except (ValueError, SyntaxError):
        pass
    
    return []

item_categories["parsed_feat"] = item_categories["feat"].apply(parse_feat)

unique_feats = set(
    feat
    for feats in item_categories["parsed_feat"]
    for feat in feats
)

print("Videos with category features:",
      (item_categories["parsed_feat"].str.len() > 0).sum())

print("Unique feature IDs:", len(unique_feats))

print("Example feature IDs:",
      sorted(unique_feats)[:20])

Videos with category features: 10728
Unique feature IDs: 31
Example feature IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


In [73]:
print("Variables currently available:")
print([
    name for name in [
        "user_interest",
        "video_tags",
        "video_tag_mapping",
        "train_data",
        "test_data"
    ]
    if name in globals()
])

Variables currently available:
['video_tags', 'video_tag_mapping', 'train_data', 'test_data']


In [74]:
# Parse detailed topic tags for the training videos

video_tags["parsed_tags"] = video_tags["topic_tag"].apply(parse_topic_tags)

# Keep only videos that actually have at least one tag
video_tags_tagged = video_tags[
    video_tags["parsed_tags"].str.len() > 0
].copy()

# One row per video-tag pair
video_tag_rows = []

for _, row in video_tags_tagged.iterrows():
    for tag in row["parsed_tags"]:
        video_tag_rows.append({
            "video_id": row["video_id"],
            "video_tag": tag
        })

video_tag_mapping = pd.DataFrame(video_tag_rows).drop_duplicates()

print("Tagged videos:",
      video_tag_mapping["video_id"].nunique())

print("Unique detailed tags:",
      video_tag_mapping["video_tag"].nunique())

print("Video-tag pairs:",
      len(video_tag_mapping))

Tagged videos: 1803
Unique detailed tags: 1921
Video-tag pairs: 4478


In [75]:
# Recreate the detailed video-tag mapping used in Notebook 03

video_tag_mapping = item_features[
    ["video_id", "video_tag_name"]
].copy()

# Keep only videos that appear in the training interactions
video_tag_mapping = video_tag_mapping[
    video_tag_mapping["video_id"].isin(
        train_data["video_id"].unique()
    )
].copy()

# Remove missing and UNKNOWN tags
video_tag_mapping = video_tag_mapping.dropna(
    subset=["video_tag_name"]
)

video_tag_mapping = video_tag_mapping[
    video_tag_mapping["video_tag_name"] != "UNKNOWN"
].copy()

# Keep one row per video-tag pair
video_tag_mapping = video_tag_mapping.drop_duplicates(
    subset=["video_id", "video_tag_name"]
)

print(
    "Videos in training:",
    train_data["video_id"].nunique()
)

print(
    "Videos with detailed tags:",
    video_tag_mapping["video_id"].nunique()
)

print(
    "Unique detailed tags:",
    video_tag_mapping["video_tag_name"].nunique()
)

print(
    "Video-tag pairs:",
    len(video_tag_mapping)
)

print("\nSample mapping:")
display(video_tag_mapping.head(10))

Videos in training: 3288
Videos with detailed tags: 3274
Unique detailed tags: 351
Video-tag pairs: 6104

Sample mapping:


,video_id,video_tag_name
5805,103,面食
5864,103,美食
6147,109,摩托车
6206,109,汽车
6785,120,搞笑
6852,122,猴
6911,122,搞笑
7273,128,美食
7328,130,侧面
7387,130,才艺


In [76]:
# Build user-level detailed interest profiles

user_tag_history = train_data[
    ["user_id", "video_id", "watch_score"]
].merge(
    video_tag_mapping,
    on="video_id",
    how="inner"
)

user_tag_profile = (
    user_tag_history
    .groupby(["user_id", "video_tag_name"])
    .agg(
        interactions=("video_id", "count"),
        avg_watch_score=("watch_score", "mean")
    )
    .reset_index()
)

print("User-tag profile rows:", len(user_tag_profile))
print("Users:", user_tag_profile["user_id"].nunique())
print("Detailed tags represented:", user_tag_profile["video_tag_name"].nunique())

print("\nExample:")
display(user_tag_profile.head(10))

User-tag profile rows: 485629
Users: 1411
Detailed tags represented: 351

Example:


,user_id,video_tag_name,interactions,avg_watch_score
0,14,一盘菜,6,0.250652
1,14,三农,8,1.190023
2,14,两性情感,2,0.919220
3,14,个性定制,4,1.192115
4,14,乌龟,1,0.330497
5,14,乐器,1,0.307241
6,14,乡村,3,1.311753
7,14,乡村女,1,1.605414
8,14,书法,46,1.023364
9,14,事故,5,0.710101


In [77]:
# Classify each user's detailed interest level

user_tag_profile["interest_level"] = np.select(
    [
        user_tag_profile["interactions"] >= 8,
        user_tag_profile["interactions"] >= 1
    ],
    [
        "Established",
        "Weak"
    ],
    default="Unseen"
)

print(
    user_tag_profile["interest_level"].value_counts()
)

print("\nUsers:",
      user_tag_profile["user_id"].nunique())

print("\nWeak interest profiles:",
      (user_tag_profile["interest_level"] == "Weak").sum())

print(
    "Established interest profiles:",
    (user_tag_profile["interest_level"] == "Established").sum()
)

interest_level
Weak           356765
Established    128864
Name: count, dtype: int64

Users: 1411

Weak interest profiles: 356765
Established interest profiles: 128864


In [78]:
# Calculate smoothed interest for each user-tag pair

user_mean = user_tag_history["watch_score"].mean()
smoothing = 5

user_tag_profile["smoothed_interest"] = (
    (
        user_tag_profile["interactions"] *
        user_tag_profile["avg_watch_score"]
    )
    +
    (smoothing * user_mean)
) / (
    user_tag_profile["interactions"] + smoothing
)

print("Overall user mean watch score:", user_mean)

print("\nSmoothed interest summary:")
display(
    user_tag_profile["smoothed_interest"].describe()
)

print("\nExample:")
display(
    user_tag_profile[
        ["user_id", "video_tag_name", "interactions",
         "avg_watch_score", "smoothed_interest", "interest_level"]
    ].head(10)
)

Overall user mean watch score: 0.8592171457777995

Smoothed interest summary:


count    485629.000000
mean          0.855018
std           0.142594
min           0.193864
25%           0.769752
50%           0.842350
75%           0.926136
max           1.780894
Name: smoothed_interest, dtype: float64


Example:


,user_id,video_tag_name,interactions,avg_watch_score,smoothed_interest,interest_level
0,14,一盘菜,6,0.250652,0.527273,Weak
1,14,三农,8,1.190023,1.062790,Established
2,14,两性情感,2,0.919220,0.876361,Weak
3,14,个性定制,4,1.192115,1.007172,Weak
4,14,乌龟,1,0.330497,0.771097,Weak
5,14,乐器,1,0.307241,0.767221,Weak
6,14,乡村,3,1.311753,1.028918,Weak
7,14,乡村女,1,1.605414,0.983583,Weak
8,14,书法,46,1.023364,1.007271,Established
9,14,事故,5,0.710101,0.784659,Weak


In [79]:
# Build exploration candidates from users' weak detailed interests

weak_interests = user_tag_profile[
    user_tag_profile["interest_level"] == "Weak"
][
    ["user_id", "video_tag_name", "smoothed_interest"]
].copy()

print("Weak user-tag profiles:", len(weak_interests))

# Match each weak interest to videos carrying that detailed tag
exploration_candidates = weak_interests.merge(
    video_tag_mapping,
    on="video_tag_name",
    how="inner"
)

print(
    "Exploration candidates before removing watched videos:",
    len(exploration_candidates)
)

# Identify videos already watched by each user
watched_pairs = train_data[
    ["user_id", "video_id"]
].drop_duplicates()

# Remove previously watched user-video pairs
exploration_candidates = exploration_candidates.merge(
    watched_pairs.assign(watched=1),
    on=["user_id", "video_id"],
    how="left"
)

exploration_candidates = exploration_candidates[
    exploration_candidates["watched"].isna()
].drop(columns=["watched"])

print(
    "Exploration candidates after removing watched videos:",
    len(exploration_candidates)
)

print(
    "Users with exploration candidates:",
    exploration_candidates["user_id"].nunique()
)

print(
    "Unique exploration videos:",
    exploration_candidates["video_id"].nunique()
)

print("\nSample exploration candidates:")

display(
    exploration_candidates[
        [
            "user_id",
            "video_id",
            "video_tag_name",
            "smoothed_interest"
        ]
    ].head(10)
)

Weak user-tag profiles: 356765
Exploration candidates before removing watched videos: 918431
Exploration candidates after removing watched videos: 48366
Users with exploration candidates: 1411
Unique exploration videos: 720

Sample exploration candidates:


,user_id,video_id,video_tag_name,smoothed_interest
22,14,9129,事故,0.784659
75,14,8624,医疗_骨科,1.119274
78,14,6308,医院,0.874974
88,14,6254,历史,0.805602
124,14,4875,国外,0.889084
163,14,3992,女-半身-清纯,0.689470
166,14,5961,女-半身-清纯,0.689470
179,14,7710,婴儿,0.772312
216,14,3024,山水风景,0.837677
224,14,3133,幼儿教育,0.861112


In [80]:
# Check how many exploration candidates each user has

exploration_per_user = (
    exploration_candidates
    .groupby("user_id")["video_id"]
    .nunique()
    .reset_index(name="exploration_candidates")
)

print("Users:", len(exploration_per_user))

print("\nExploration candidates per user:")
display(
    exploration_per_user["exploration_candidates"].describe()
)

print("\nUsers with fewer than 10 candidates:",
      (exploration_per_user["exploration_candidates"] < 10).sum())

print("Users with at least 10 candidates:",
      (exploration_per_user["exploration_candidates"] >= 10).sum())

print("\nMinimum candidates:",
      exploration_per_user["exploration_candidates"].min())

print("Maximum candidates:",
      exploration_per_user["exploration_candidates"].max())

Users: 1411

Exploration candidates per user:


count    1411.000000
mean       33.878809
std        11.585766
min         6.000000
25%        25.000000
50%        34.000000
75%        42.000000
max        78.000000
Name: exploration_candidates, dtype: float64


Users with fewer than 10 candidates: 7
Users with at least 10 candidates: 1404

Minimum candidates: 6
Maximum candidates: 78


In [81]:
# Rank exploration candidates by detailed-interest relevance

exploration_candidates = exploration_candidates.sort_values(
    ["user_id", "smoothed_interest"],
    ascending=[True, False]
).reset_index(drop=True)

print("Exploration candidates:", len(exploration_candidates))

print("\nExploration score summary:")
display(
    exploration_candidates["smoothed_interest"].describe()
)

print("\nTop exploration candidates for User 14:")

display(
    exploration_candidates[
        exploration_candidates["user_id"] == 14
    ][
        ["user_id", "video_id", "video_tag_name", "smoothed_interest"]
    ].head(10)
)

Exploration candidates: 48366

Exploration score summary:


count    48366.000000
mean         0.837906
std          0.154168
min          0.406023
25%          0.746735
50%          0.831300
75%          0.921657
max          1.692356
Name: smoothed_interest, dtype: float64


Top exploration candidates for User 14:


,user_id,video_id,video_tag_name,smoothed_interest
0,14,8624,医疗_骨科,1.119274
1,14,835,育儿知识,1.104619
2,14,2365,育儿知识,1.104619
3,14,2102,祝福,1.103374
4,14,7703,温馨,1.061058
5,14,9986,电梯,1.013739
6,14,4712,瑜伽,0.977430
7,14,6017,篮球,0.973309
8,14,1180,床,0.940659
9,14,628,心灵鸡汤,0.930359


In [82]:
# Build the normal interest-based candidate pool
# using established + weak detailed interests

# Use every known user-tag interest
known_interests = user_tag_profile[
    ["user_id", "video_tag_name", "smoothed_interest"]
].copy()

# Match user interests to videos with those tags
normal_candidates = known_interests.merge(
    video_tag_mapping,
    on="video_tag_name",
    how="inner"
)

# Remove videos already watched during training
normal_candidates = normal_candidates.merge(
    watched_pairs.assign(watched=1),
    on=["user_id", "video_id"],
    how="left"
)

normal_candidates = normal_candidates[
    normal_candidates["watched"].isna()
].drop(columns=["watched"])

# If a video has multiple matching tags for the same user,
# keep its strongest interest score
normal_candidates = (
    normal_candidates
    .groupby(["user_id", "video_id"], as_index=False)
    .agg(
        interest_score=("smoothed_interest", "max")
    )
)

print("Normal unseen candidates:", len(normal_candidates))
print("Users:", normal_candidates["user_id"].nunique())
print("Videos:", normal_candidates["video_id"].nunique())

print("\nInterest score summary:")
display(normal_candidates["interest_score"].describe())

print("\nSample:")
display(normal_candidates.head(10))

Normal unseen candidates: 250461
Users: 1411
Videos: 3212

Interest score summary:


count    250461.000000
mean          0.895469
std           0.176824
min           0.193864
25%           0.779589
50%           0.886301
75%           1.002290
max           1.762726
Name: interest_score, dtype: float64


Sample:


,user_id,video_id,interest_score
0,14,152,1.161726
1,14,365,1.089766
2,14,418,0.991058
3,14,628,0.930359
4,14,694,0.713761
5,14,770,0.975749
6,14,835,1.104619
7,14,898,0.859267
8,14,1180,0.972002
9,14,1268,0.934144


In [83]:
# Create recommendation lists with controlled exploration rates

exploration_rates = [0.00, 0.10, 0.20, 0.30]

controlled_recommendations = {}

for rate in exploration_rates:

    n_explore = int(10 * rate)
    n_normal = 10 - n_explore

    recommendations = []

    for user_id in train_data["user_id"].unique():

        # -----------------------------
        # Normal recommendations
        # -----------------------------
        user_normal = (
            normal_candidates[
                normal_candidates["user_id"] == user_id
            ]
            .sort_values("interest_score", ascending=False)
            .head(n_normal)
        )

        # -----------------------------
        # Exploration recommendations
        # -----------------------------
        user_explore = (
            exploration_candidates[
                exploration_candidates["user_id"] == user_id
            ]
            .sort_values("smoothed_interest", ascending=False)
            .head(n_explore)
        )

        # Add normal recommendations
        for _, row in user_normal.iterrows():
            recommendations.append({
                "user_id": user_id,
                "video_id": row["video_id"],
                "recommendation_type": "Normal"
            })

        # Add exploration recommendations
        for _, row in user_explore.iterrows():
            recommendations.append({
                "user_id": user_id,
                "video_id": row["video_id"],
                "recommendation_type": "Exploration"
            })

    rec_df = pd.DataFrame(recommendations)

    controlled_recommendations[rate] = rec_df

    print(
        f"Exploration {int(rate * 100)}%:"
    )
    print(
        "Total recommendations:",
        len(rec_df)
    )
    print(
        rec_df["recommendation_type"].value_counts().to_dict()
    )
    print()

Exploration 0%:
Total recommendations: 14110
{'Normal': 14110}

Exploration 10%:
Total recommendations: 14110
{'Normal': 12699, 'Exploration': 1411}

Exploration 20%:
Total recommendations: 14110
{'Normal': 11288, 'Exploration': 2822}

Exploration 30%:
Total recommendations: 14110
{'Normal': 9877, 'Exploration': 4233}



In [84]:
# Create recommendation lists with controlled exploration rates

exploration_rates = [0.00, 0.10, 0.20, 0.30]

controlled_recommendations = {}

for rate in exploration_rates:

    n_explore = int(10 * rate)
    n_normal = 10 - n_explore

    recommendations = []

    for user_id in train_data["user_id"].unique():

        # -----------------------------
        # Normal recommendations
        # -----------------------------
        user_normal = (
            normal_candidates[
                normal_candidates["user_id"] == user_id
            ]
            .sort_values("interest_score", ascending=False)
            .head(n_normal)
        )

        # -----------------------------
        # Exploration recommendations
        # -----------------------------
        user_explore = (
            exploration_candidates[
                exploration_candidates["user_id"] == user_id
            ]
            .sort_values("smoothed_interest", ascending=False)
            .head(n_explore)
        )

        # Add normal recommendations
        for _, row in user_normal.iterrows():
            recommendations.append({
                "user_id": user_id,
                "video_id": row["video_id"],
                "recommendation_type": "Normal"
            })

        # Add exploration recommendations
        for _, row in user_explore.iterrows():
            recommendations.append({
                "user_id": user_id,
                "video_id": row["video_id"],
                "recommendation_type": "Exploration"
            })

    rec_df = pd.DataFrame(recommendations)

    controlled_recommendations[rate] = rec_df

    print(
        f"Exploration {int(rate * 100)}%:"
    )
    print(
        "Total recommendations:",
        len(rec_df)
    )
    print(
        rec_df["recommendation_type"].value_counts().to_dict()
    )
    print()
    

Exploration 0%:
Total recommendations: 14110
{'Normal': 14110}

Exploration 10%:
Total recommendations: 14110
{'Normal': 12699, 'Exploration': 1411}

Exploration 20%:
Total recommendations: 14110
{'Normal': 11288, 'Exploration': 2822}

Exploration 30%:
Total recommendations: 14110
{'Normal': 9877, 'Exploration': 4233}



In [85]:
# Evaluate controlled exploration recommendations
# against the held-out test period

# Test interactions
test_pairs = test_data[
    ["user_id", "video_id"]
].drop_duplicates()

# Mark every test interaction
test_pairs["test_hit"] = 1

evaluation_results = []

for rate, rec_df in controlled_recommendations.items():

    # Add test-period interaction information
    evaluated = rec_df.merge(
        test_pairs,
        on=["user_id", "video_id"],
        how="left"
    )

    evaluated["test_hit"] = (
        evaluated["test_hit"]
        .fillna(0)
        .astype(int)
    )

    # --------------------------------
    # Overall HitRate@10
    # --------------------------------
    user_hits = (
        evaluated
        .groupby("user_id")["test_hit"]
        .max()
    )

    overall_hit_rate = user_hits.mean()

    # --------------------------------
    # Normal recommendation hits
    # --------------------------------
    normal_recs = evaluated[
        evaluated["recommendation_type"] == "Normal"
    ]

    normal_hits = normal_recs["test_hit"].sum()

    # --------------------------------
    # Exploration recommendation hits
    # --------------------------------
    exploration_recs = evaluated[
        evaluated["recommendation_type"] == "Exploration"
    ]

    exploration_hits = exploration_recs["test_hit"].sum()

    if len(exploration_recs) > 0:
        exploration_hit_rate = (
            exploration_hits / len(exploration_recs)
        )
    else:
        exploration_hit_rate = np.nan

    # --------------------------------
    # Users receiving >=1 hit
    # --------------------------------
    users_with_hit = (
        (user_hits >= 1).sum()
    )

    evaluation_results.append({
        "exploration_rate": rate,
        "total_recommendations": len(evaluated),
        "overall_HitRate@10": overall_hit_rate,
        "users_with_at_least_one_hit": users_with_hit,
        "normal_hits": normal_hits,
        "exploration_recommendations": len(exploration_recs),
        "exploration_hits": exploration_hits,
        "exploration_hit_rate": exploration_hit_rate
    })

evaluation_summary = pd.DataFrame(evaluation_results)

display(evaluation_summary)

,exploration_rate,total_recommendations,overall_HitRate@10,users_with_at_least_one_hit,normal_hits,exploration_recommendations,exploration_hits,exploration_hit_rate
0,0.0,14110,0.892984,1260,3204,0,0,NaN
1,0.1,14110,0.863926,1219,2850,1411,276,0.195606
2,0.2,14110,0.836995,1181,2480,2822,581,0.205882
3,0.3,14110,0.820695,1158,2159,4233,867,0.204819


In [86]:
# Rebuild creator popularity statistics and creator groups

# Sum show counts across all available daily records
creator_stats = (
    item_features
    .groupby("author_id")["show_cnt"]
    .sum()
    .reset_index(name="total_shows")
)

# Use the same Emerging threshold established earlier:
# bottom 25% of creators by total_shows
emerging_threshold = creator_stats["total_shows"].quantile(0.25)

creator_stats["creator_group"] = np.where(
    creator_stats["total_shows"] <= emerging_threshold,
    "Emerging",
    "Established"
)

print("Emerging threshold:", emerging_threshold)

print("\nCreator groups:")
print(
    creator_stats["creator_group"].value_counts()
)

print("\nCreator statistics:")
display(
    creator_stats["total_shows"].describe()
)

Emerging threshold: 4354.0

Creator groups:
creator_group
Established    6275
Emerging       2094
Name: count, dtype: int64

Creator statistics:


count    8.369000e+03
mean     3.866465e+06
std      1.452844e+07
min      3.000000e+00
25%      4.354000e+03
50%      4.709300e+04
75%      8.410930e+05
max      3.750626e+08
Name: total_shows, dtype: float64

In [87]:
# Rebuild creator groups using only creators represented
# by videos in the interaction dataset

# Videos that actually appear in the interaction dataset
experiment_videos = small_matrix["video_id"].unique()

# Get their creators
experiment_creator_mapping = item_features[
    ["video_id", "author_id", "show_cnt"]
].copy()

experiment_creator_mapping = experiment_creator_mapping[
    experiment_creator_mapping["video_id"].isin(experiment_videos)
].copy()

# Sum daily show counts only for experiment videos
experiment_creator_stats = (
    experiment_creator_mapping
    .groupby("author_id")["show_cnt"]
    .sum()
    .reset_index(name="total_shows")
)

# Bottom 25% = Emerging
experiment_threshold = (
    experiment_creator_stats["total_shows"]
    .quantile(0.25)
)

experiment_creator_stats["creator_group"] = np.where(
    experiment_creator_stats["total_shows"] <= experiment_threshold,
    "Emerging",
    "Established"
)

print("Experiment creators:",
      len(experiment_creator_stats))

print("\nEmerging threshold:",
      experiment_threshold)

print("\nCreator groups:")
print(
    experiment_creator_stats["creator_group"].value_counts()
)

Experiment creators: 2031

Emerging threshold: 1073502.5

Creator groups:
creator_group
Established    1523
Emerging        508
Name: count, dtype: int64


In [88]:
# Evaluate creator exposure for each exploration condition

creator_mapping = experiment_creator_mapping[
    ["video_id", "author_id"]
].drop_duplicates(
    subset=["video_id"]
)

creator_mapping = creator_mapping.merge(
    experiment_creator_stats[
        ["author_id", "creator_group"]
    ],
    on="author_id",
    how="left"
)

creator_exposure_results = []

for rate, rec_df in controlled_recommendations.items():

    evaluated = rec_df.merge(
        creator_mapping,
        on="video_id",
        how="left"
    )

    # Count recommendation exposure
    group_counts = evaluated["creator_group"].value_counts()

    emerging_exposure = group_counts.get("Emerging", 0)
    established_exposure = group_counts.get("Established", 0)

    total_exposure = (
        emerging_exposure +
        established_exposure
    )

    emerging_share = (
        emerging_exposure / total_exposure
        if total_exposure > 0 else np.nan
    )

    established_share = (
        established_exposure / total_exposure
        if total_exposure > 0 else np.nan
    )

    # Unique creators exposed
    unique_creators = evaluated["author_id"].nunique()

    # Emerging creator coverage
    emerging_creators_exposed = (
        evaluated.loc[
            evaluated["creator_group"] == "Emerging",
            "author_id"
        ].nunique()
    )

    # Established creator coverage
    established_creators_exposed = (
        evaluated.loc[
            evaluated["creator_group"] == "Established",
            "author_id"
        ].nunique()
    )

    creator_exposure_results.append({
        "exploration_rate": rate,
        "emerging_exposure": emerging_exposure,
        "established_exposure": established_exposure,
        "emerging_share": emerging_share,
        "established_share": established_share,
        "unique_creators_exposed": unique_creators,
        "emerging_creators_exposed": emerging_creators_exposed,
        "established_creators_exposed": established_creators_exposed
    })

creator_exposure_summary = pd.DataFrame(
    creator_exposure_results
)

display(creator_exposure_summary)

,exploration_rate,emerging_exposure,established_exposure,emerging_share,established_share,unique_creators_exposed,emerging_creators_exposed,established_creators_exposed
0,0.0,1422,12688,0.100780,0.899220,1069,240,829
1,0.1,1422,12688,0.100780,0.899220,1038,229,809
2,0.2,1425,12685,0.100992,0.899008,1013,229,784
3,0.3,1400,12710,0.099220,0.900780,980,222,758


In [89]:
# Evaluate creator exposure for each exploration condition

creator_mapping = experiment_creator_mapping[
    ["video_id", "author_id"]
].drop_duplicates(
    subset=["video_id"]
)

creator_mapping = creator_mapping.merge(
    experiment_creator_stats[
        ["author_id", "creator_group"]
    ],
    on="author_id",
    how="left"
)

creator_exposure_results = []

for rate, rec_df in controlled_recommendations.items():

    evaluated = rec_df.merge(
        creator_mapping,
        on="video_id",
        how="left"
    )

    # Count recommendation exposure
    group_counts = evaluated["creator_group"].value_counts()

    emerging_exposure = group_counts.get("Emerging", 0)
    established_exposure = group_counts.get("Established", 0)

    total_exposure = (
        emerging_exposure +
        established_exposure
    )

    emerging_share = (
        emerging_exposure / total_exposure
        if total_exposure > 0 else np.nan
    )

    established_share = (
        established_exposure / total_exposure
        if total_exposure > 0 else np.nan
    )

    # Unique creators exposed
    unique_creators = evaluated["author_id"].nunique()

    # Emerging creator coverage
    emerging_creators_exposed = (
        evaluated.loc[
            evaluated["creator_group"] == "Emerging",
            "author_id"
        ].nunique()
    )

    # Established creator coverage
    established_creators_exposed = (
        evaluated.loc[
            evaluated["creator_group"] == "Established",
            "author_id"
        ].nunique()
    )

    creator_exposure_results.append({
        "exploration_rate": rate,
        "emerging_exposure": emerging_exposure,
        "established_exposure": established_exposure,
        "emerging_share": emerging_share,
        "established_share": established_share,
        "unique_creators_exposed": unique_creators,
        "emerging_creators_exposed": emerging_creators_exposed,
        "established_creators_exposed": established_creators_exposed
    })

creator_exposure_summary = pd.DataFrame(
    creator_exposure_results
)

display(creator_exposure_summary)

,exploration_rate,emerging_exposure,established_exposure,emerging_share,established_share,unique_creators_exposed,emerging_creators_exposed,established_creators_exposed
0,0.0,1422,12688,0.100780,0.899220,1069,240,829
1,0.1,1422,12688,0.100780,0.899220,1038,229,809
2,0.2,1425,12685,0.100992,0.899008,1013,229,784
3,0.3,1400,12710,0.099220,0.900780,980,222,758


In [90]:
# Calculate average creator diversity per user's Top-10

diversity_results = []

for rate, rec_df in controlled_recommendations.items():

    evaluated = rec_df.merge(
        creator_mapping,
        on="video_id",
        how="left"
    )

    creators_per_user = (
        evaluated
        .groupby("user_id")["author_id"]
        .nunique()
    )

    diversity_results.append({
        "exploration_rate": rate,
        "mean_unique_creators_per_user": creators_per_user.mean(),
        "median_unique_creators_per_user": creators_per_user.median(),
        "minimum_unique_creators": creators_per_user.min(),
        "maximum_unique_creators": creators_per_user.max()
    })

creator_diversity_summary = pd.DataFrame(
    diversity_results
)

display(creator_diversity_summary)

,exploration_rate,mean_unique_creators_per_user,median_unique_creators_per_user,minimum_unique_creators,maximum_unique_creators
0,0.0,9.835578,10.0,7,10
1,0.1,9.017009,9.0,7,10
2,0.2,8.398299,8.0,6,10
3,0.3,8.002126,8.0,6,10


In [91]:
# Apply fairness boost to exploration candidates

fairness_strength = 0.08

fair_exploration = exploration_candidates.copy()

# Add creator information
fair_exploration = fair_exploration.merge(
    creator_mapping[
        ["video_id", "author_id", "creator_group"]
    ],
    on="video_id",
    how="left"
)

# Give Emerging creators an additional score
fair_exploration["fairness_boost"] = np.where(
    fair_exploration["creator_group"] == "Emerging",
    fairness_strength,
    0
)

# Calculate combined score
fair_exploration["combined_exploration_score"] = (
    fair_exploration["smoothed_interest"]
    + fair_exploration["fairness_boost"]
)

# Rank candidates for each user
fair_exploration = fair_exploration.sort_values(
    ["user_id", "combined_exploration_score"],
    ascending=[True, False]
).reset_index(drop=True)

print("Fairness-aware candidates:", len(fair_exploration))

print(
    "Emerging candidates:",
    (fair_exploration["creator_group"] == "Emerging").sum()
)

print(
    "Established candidates:",
    (fair_exploration["creator_group"] == "Established").sum()
)

print(
    "Missing creator groups:",
    fair_exploration["creator_group"].isna().sum()
)

print("\nUser 14 top fairness-aware exploration candidates:")

display(
    fair_exploration[
        fair_exploration["user_id"] == 14
    ][
        [
            "user_id",
            "video_id",
            "video_tag_name",
            "smoothed_interest",
            "creator_group",
            "fairness_boost",
            "combined_exploration_score"
        ]
    ].head(10)
)

Fairness-aware candidates: 48366
Emerging candidates: 4891
Established candidates: 43475
Missing creator groups: 0

User 14 top fairness-aware exploration candidates:


,user_id,video_id,video_tag_name,smoothed_interest,creator_group,fairness_boost,combined_exploration_score
0,14,8624,医疗_骨科,1.119274,Established,0.00,1.119274
1,14,835,育儿知识,1.104619,Established,0.00,1.104619
2,14,2365,育儿知识,1.104619,Established,0.00,1.104619
3,14,2102,祝福,1.103374,Established,0.00,1.103374
4,14,9986,电梯,1.013739,Emerging,0.08,1.093739
5,14,7703,温馨,1.061058,Established,0.00,1.061058
6,14,628,心灵鸡汤,0.930359,Emerging,0.08,1.010359
7,14,4712,瑜伽,0.977430,Established,0.00,0.977430
8,14,6017,篮球,0.973309,Established,0.00,0.973309
9,14,1180,床,0.940659,Established,0.00,0.940659


In [92]:
# Select top 3 fairness-aware exploration videos per user

fair_exploration_recommendations = (
    fair_exploration
    .sort_values(
        ["user_id", "combined_exploration_score"],
        ascending=[True, False]
    )
    .groupby("user_id")
    .head(3)
    .copy()
)

fair_exploration_recommendations["recommendation_type"] = (
    "Fairness-aware Exploration"
)

print(
    "Total recommendations:",
    len(fair_exploration_recommendations)
)

print(
    "Users:",
    fair_exploration_recommendations["user_id"].nunique()
)

print("\nCreator group exposure:")

print(
    fair_exploration_recommendations[
        "creator_group"
    ].value_counts()
)

print("\nSample recommendations:")
display(
    fair_exploration_recommendations[
        [
            "user_id",
            "video_id",
            "creator_group",
            "smoothed_interest",
            "combined_exploration_score"
        ]
    ].head(10)
)

Total recommendations: 4233
Users: 1411

Creator group exposure:
creator_group
Established    3405
Emerging        828
Name: count, dtype: int64

Sample recommendations:


,user_id,video_id,creator_group,smoothed_interest,combined_exploration_score
0,14,8624,Established,1.119274,1.119274
1,14,835,Established,1.104619,1.104619
2,14,2365,Established,1.104619,1.104619
29,19,4050,Established,0.968924,0.968924
30,19,8361,Established,0.968924,0.968924
31,19,2588,Emerging,0.886046,0.966046
64,21,10405,Established,1.137878,1.137878
65,21,9109,Established,1.100608,1.100608
66,21,3102,Established,1.086823,1.086823
94,23,3247,Established,1.383505,1.383505


In [93]:
# Evaluate fairness-aware exploration using test interactions

fair_evaluated = fair_exploration_recommendations.merge(
    test_pairs,
    on=["user_id", "video_id"],
    how="left"
)

fair_evaluated["test_hit"] = (
    fair_evaluated["test_hit"]
    .fillna(0)
    .astype(int)
)

# Overall exploration acceptance
total_fair_recommendations = len(fair_evaluated)
total_fair_hits = fair_evaluated["test_hit"].sum()

fair_acceptance_rate = (
    total_fair_hits / total_fair_recommendations
)

# Acceptance by creator group
fair_group_acceptance = (
    fair_evaluated
    .groupby("creator_group")
    .agg(
        recommendations=("test_hit", "count"),
        accepted=("test_hit", "sum")
    )
    .reset_index()
)

fair_group_acceptance["acceptance_rate"] = (
    fair_group_acceptance["accepted"] /
    fair_group_acceptance["recommendations"]
)

print("Total recommendations:", total_fair_recommendations)
print("Test-period hits:", total_fair_hits)

print(
    "Overall exploration acceptance:",
    round(fair_acceptance_rate * 100, 2), "%"
)

print("\nAcceptance by creator group:")
display(fair_group_acceptance)

Total recommendations: 4233
Test-period hits: 789
Overall exploration acceptance: 18.64 %

Acceptance by creator group:


,creator_group,recommendations,accepted,acceptance_rate
0,Emerging,828,67,0.080918
1,Established,3405,722,0.212041


In [94]:
# Construct complete fairness-aware 30% exploration recommendations

# Keep the 7 normal recommendations from the 30% condition
normal_30 = controlled_recommendations[0.30]

normal_30 = normal_30[
    normal_30["recommendation_type"] == "Normal"
].copy()

# Select the fairness-aware exploration recommendations
fair_explore_30 = fair_exploration_recommendations[
    ["user_id", "video_id"]
].copy()

fair_explore_30["recommendation_type"] = "Exploration"

# Combine both parts
fairness_aware_top10 = pd.concat(
    [normal_30, fair_explore_30],
    ignore_index=True
)

# Evaluate against held-out test interactions
fairness_aware_evaluated = fairness_aware_top10.merge(
    test_pairs[["user_id", "video_id"]],
    on=["user_id", "video_id"],
    how="left",
    indicator=True
)

fairness_aware_evaluated["test_hit"] = (
    fairness_aware_evaluated["_merge"] == "both"
).astype(int)

# Overall HitRate@10
fair_user_hits = (
    fairness_aware_evaluated
    .groupby("user_id")["test_hit"]
    .max()
)

fair_top10_hit_rate = fair_user_hits.mean()

# Creator exposure
fairness_aware_evaluated = fairness_aware_evaluated.merge(
    creator_mapping[
        ["video_id", "author_id", "creator_group"]
    ],
    on="video_id",
    how="left"
)

fair_group_counts = (
    fairness_aware_evaluated["creator_group"]
    .value_counts()
)

fair_emerging_share = (
    fair_group_counts.get("Emerging", 0) /
    len(fairness_aware_evaluated)
)

# Creator diversity per user
fair_creators_per_user = (
    fairness_aware_evaluated
    .groupby("user_id")["author_id"]
    .nunique()
)

print("Total recommendations:", len(fairness_aware_top10))
print("Users:", fairness_aware_top10["user_id"].nunique())

print("\nOverall HitRate@10:",
      round(fair_top10_hit_rate, 4))

print("Users with at least one hit:",
      int((fair_user_hits >= 1).sum()))

print("\nEmerging exposure:",
      fair_group_counts.get("Emerging", 0))

print("Established exposure:",
      fair_group_counts.get("Established", 0))

print("Emerging exposure share:",
      round(fair_emerging_share * 100, 2), "%")

print("\nMean unique creators per user:",
      round(fair_creators_per_user.mean(), 4))

Total recommendations: 14110
Users: 1411

Overall HitRate@10: 0.8157
Users with at least one hit: 1151

Emerging exposure: 1846
Established exposure: 12264
Emerging exposure share: 13.08 %

Mean unique creators per user: 8.0921


In [95]:
# Check for duplicate user-video recommendations

print("Normal and exploration candidate overlap:")

candidate_overlap = normal_candidates.merge(
    exploration_candidates[["user_id", "video_id"]],
    on=["user_id", "video_id"],
    how="inner"
)

print("Overlapping user-video pairs:", len(candidate_overlap))

print("\nControlled recommendation validation:")

for rate, rec_df in controlled_recommendations.items():

    duplicate_pairs = rec_df.duplicated(
        subset=["user_id", "video_id"]
    ).sum()

    recommendations_per_user = (
        rec_df.groupby("user_id")["video_id"].nunique()
    )

    print(f"\nExploration rate: {int(rate * 100)}%")
    print("Duplicate pairs:", duplicate_pairs)
    print("Minimum unique videos per user:",
          recommendations_per_user.min())
    print("Maximum unique videos per user:",
          recommendations_per_user.max())

print("\nFairness-aware Top-10 validation:")

fair_duplicates = fairness_aware_top10.duplicated(
    subset=["user_id", "video_id"]
).sum()

fair_unique_per_user = (
    fairness_aware_top10.groupby("user_id")["video_id"].nunique()
)

print("Duplicate pairs:", fair_duplicates)
print("Minimum unique videos per user:", fair_unique_per_user.min())
print("Maximum unique videos per user:", fair_unique_per_user.max())

Normal and exploration candidate overlap:
Overlapping user-video pairs: 48366

Controlled recommendation validation:

Exploration rate: 0%
Duplicate pairs: 0
Minimum unique videos per user: 10
Maximum unique videos per user: 10

Exploration rate: 10%
Duplicate pairs: 1196
Minimum unique videos per user: 9
Maximum unique videos per user: 10

Exploration rate: 20%
Duplicate pairs: 2101
Minimum unique videos per user: 8
Maximum unique videos per user: 10

Exploration rate: 30%
Duplicate pairs: 2674
Minimum unique videos per user: 7
Maximum unique videos per user: 10

Fairness-aware Top-10 validation:
Duplicate pairs: 2550
Minimum unique videos per user: 7
Maximum unique videos per user: 10


In [96]:
# Rebuild mutually exclusive normal and exploration pools

# ----------------------------------
# 1. Established-interest candidates
# ----------------------------------

established_interests = user_tag_profile[
    user_tag_profile["interest_level"] == "Established"
][
    ["user_id", "video_tag_name", "smoothed_interest"]
].copy()

normal_candidates = established_interests.merge(
    video_tag_mapping,
    on="video_tag_name",
    how="inner"
)

# Remove previously watched videos
normal_candidates = normal_candidates.merge(
    watched_pairs.assign(watched=1),
    on=["user_id", "video_id"],
    how="left"
)

normal_candidates = normal_candidates[
    normal_candidates["watched"].isna()
].drop(columns=["watched"])

# Keep one row per user-video pair
normal_candidates = (
    normal_candidates
    .groupby(["user_id", "video_id"], as_index=False)
    .agg(interest_score=("smoothed_interest", "max"))
)

# ----------------------------------
# 2. Weak-interest exploration candidates
# ----------------------------------

weak_interests = user_tag_profile[
    user_tag_profile["interest_level"] == "Weak"
][
    ["user_id", "video_tag_name", "smoothed_interest"]
].copy()

exploration_candidates = weak_interests.merge(
    video_tag_mapping,
    on="video_tag_name",
    how="inner"
)

exploration_candidates = exploration_candidates.merge(
    watched_pairs.assign(watched=1),
    on=["user_id", "video_id"],
    how="left"
)

exploration_candidates = exploration_candidates[
    exploration_candidates["watched"].isna()
].drop(columns=["watched"])

# Keep one row per user-video pair
exploration_candidates = (
    exploration_candidates
    .groupby(["user_id", "video_id"], as_index=False)
    .agg(smoothed_interest=("smoothed_interest", "max"))
)

# ----------------------------------
# 3. Validate separation
# ----------------------------------

overlap = normal_candidates.merge(
    exploration_candidates,
    on=["user_id", "video_id"],
    how="inner"
)

print("Normal candidates:", len(normal_candidates))
print("Exploration candidates:", len(exploration_candidates))

print("\nNormal users:",
      normal_candidates["user_id"].nunique())

print("Exploration users:",
      exploration_candidates["user_id"].nunique())

print("\nOverlapping user-video pairs:", len(overlap))

print("\nNormal candidates per user:")
display(
    normal_candidates.groupby("user_id")["video_id"].nunique().describe()
)

print("\nExploration candidates per user:")
display(
    exploration_candidates.groupby("user_id")["video_id"].nunique().describe()
)

Normal candidates: 247401
Exploration candidates: 47803

Normal users: 1411
Exploration users: 1411

Overlapping user-video pairs: 44743

Normal candidates per user:


count    1411.000000
mean      175.337349
std        52.349056
min        48.000000
25%       137.000000
50%       176.000000
75%       213.500000
max       369.000000
Name: video_id, dtype: float64


Exploration candidates per user:


count    1411.000000
mean       33.878809
std        11.585766
min         6.000000
25%        25.000000
50%        34.000000
75%        42.000000
max        78.000000
Name: video_id, dtype: float64

In [97]:
# Remove videos already eligible as normal recommendations
# from the exploration pool

normal_pairs = normal_candidates[
    ["user_id", "video_id"]
].drop_duplicates()

exploration_candidates = exploration_candidates.merge(
    normal_pairs.assign(is_normal=1),
    on=["user_id", "video_id"],
    how="left"
)

exploration_candidates = exploration_candidates[
    exploration_candidates["is_normal"].isna()
].drop(columns=["is_normal"])

# Verify that the two pools are now mutually exclusive
remaining_overlap = normal_candidates.merge(
    exploration_candidates,
    on=["user_id", "video_id"],
    how="inner"
)

print("Normal candidates:", len(normal_candidates))
print("Exploration candidates:", len(exploration_candidates))

print("\nRemaining overlapping pairs:", len(remaining_overlap))

print("\nExploration candidates per user:")
display(
    exploration_candidates
    .groupby("user_id")["video_id"]
    .nunique()
    .describe()
)

print(
    "\nUsers with at least 3 exploration candidates:",
    (
        exploration_candidates
        .groupby("user_id")["video_id"]
        .nunique() >= 3
    ).sum()
)

Normal candidates: 247401
Exploration candidates: 3060

Remaining overlapping pairs: 0

Exploration candidates per user:


count    1187.000000
mean        2.577928
std         1.527583
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max         9.000000
Name: video_id, dtype: float64


Users with at least 3 exploration candidates: 505


In [98]:
# Restore complete, independent candidate pools

# 1. Normal candidates: all known interests

known_interests = user_tag_profile[
    ["user_id", "video_tag_name", "smoothed_interest"]
].copy()

normal_candidates = known_interests.merge(
    video_tag_mapping,
    on="video_tag_name",
    how="inner"
)

normal_candidates = normal_candidates.merge(
    watched_pairs.assign(watched=1),
    on=["user_id", "video_id"],
    how="left"
)

normal_candidates = normal_candidates[
    normal_candidates["watched"].isna()
].drop(columns=["watched"])

normal_candidates = (
    normal_candidates
    .groupby(["user_id", "video_id"], as_index=False)
    .agg(interest_score=("smoothed_interest", "max"))
)

# 2. Exploration candidates: weak interests only

weak_interests = user_tag_profile[
    user_tag_profile["interest_level"] == "Weak"
][
    ["user_id", "video_tag_name", "smoothed_interest"]
].copy()

exploration_candidates = weak_interests.merge(
    video_tag_mapping,
    on="video_tag_name",
    how="inner"
)

exploration_candidates = exploration_candidates.merge(
    watched_pairs.assign(watched=1),
    on=["user_id", "video_id"],
    how="left"
)

exploration_candidates = exploration_candidates[
    exploration_candidates["watched"].isna()
].drop(columns=["watched"])

exploration_candidates = (
    exploration_candidates
    .groupby(["user_id", "video_id"], as_index=False)
    .agg(smoothed_interest=("smoothed_interest", "max"))
)

# 3. Check candidate availability

normal_counts = normal_candidates.groupby("user_id")["video_id"].nunique()

exploration_counts = exploration_candidates.groupby("user_id")["video_id"].nunique()

print("Normal candidates:", len(normal_candidates))
print("Exploration candidates:", len(exploration_candidates))

print("\nUsers with normal candidates:", len(normal_counts))
print("Users with exploration candidates:", len(exploration_counts))

print("\nMinimum normal candidates:", normal_counts.min())
print("Minimum exploration candidates:", exploration_counts.min())

print("\nUsers with at least 3 exploration candidates:",
      (exploration_counts >= 3).sum())

Normal candidates: 250461
Exploration candidates: 47803

Users with normal candidates: 1411
Users with exploration candidates: 1411

Minimum normal candidates: 48
Minimum exploration candidates: 6

Users with at least 3 exploration candidates: 1411


In [99]:
# Rebuild controlled recommendations without duplicates

exploration_rates = [0.00, 0.10, 0.20, 0.30]

controlled_recommendations = {}

for rate in exploration_rates:

    n_explore = int(10 * rate)
    n_normal = 10 - n_explore

    recommendations = []

    for user_id in train_data["user_id"].unique():

        # Select exploration candidates first
        user_explore = (
            exploration_candidates[
                exploration_candidates["user_id"] == user_id
            ]
            .sort_values("smoothed_interest", ascending=False)
            .head(n_explore)
        )

        selected_explore_ids = set(user_explore["video_id"])

        # Select normal candidates excluding exploration videos
        user_normal = normal_candidates[
            (normal_candidates["user_id"] == user_id) &
            (~normal_candidates["video_id"].isin(selected_explore_ids))
        ].sort_values(
            "interest_score",
            ascending=False
        ).head(n_normal)

        # Add exploration recommendations
        for _, row in user_explore.iterrows():
            recommendations.append({
                "user_id": user_id,
                "video_id": row["video_id"],
                "recommendation_type": "Exploration"
            })

        # Add normal recommendations
        for _, row in user_normal.iterrows():
            recommendations.append({
                "user_id": user_id,
                "video_id": row["video_id"],
                "recommendation_type": "Normal"
            })

    rec_df = pd.DataFrame(recommendations)

    controlled_recommendations[rate] = rec_df

    # Validate
    unique_per_user = (
        rec_df.groupby("user_id")["video_id"].nunique()
    )

    print(f"\nExploration: {int(rate * 100)}%")
    print("Total recommendations:", len(rec_df))
    print("Users:", rec_df["user_id"].nunique())
    print("Duplicate pairs:",
          rec_df.duplicated(["user_id", "video_id"]).sum())
    print("Minimum unique videos/user:", unique_per_user.min())
    print("Maximum unique videos/user:", unique_per_user.max())
    print("Recommendation types:",
          rec_df["recommendation_type"].value_counts().to_dict())


Exploration: 0%
Total recommendations: 14110
Users: 1411
Duplicate pairs: 0
Minimum unique videos/user: 10
Maximum unique videos/user: 10
Recommendation types: {'Normal': 14110}

Exploration: 10%
Total recommendations: 14110
Users: 1411
Duplicate pairs: 0
Minimum unique videos/user: 10
Maximum unique videos/user: 10
Recommendation types: {'Normal': 12699, 'Exploration': 1411}

Exploration: 20%
Total recommendations: 14110
Users: 1411
Duplicate pairs: 0
Minimum unique videos/user: 10
Maximum unique videos/user: 10
Recommendation types: {'Normal': 11288, 'Exploration': 2822}

Exploration: 30%
Total recommendations: 14110
Users: 1411
Duplicate pairs: 0
Minimum unique videos/user: 10
Maximum unique videos/user: 10
Recommendation types: {'Normal': 9877, 'Exploration': 4233}


In [100]:
# Evaluate recommendation quality

test_pairs = test_data[
    ["user_id", "video_id"]
].drop_duplicates().copy()

test_pairs["test_hit"] = 1

evaluation_results = []

for rate, rec_df in controlled_recommendations.items():

    evaluated = rec_df.merge(
        test_pairs,
        on=["user_id", "video_id"],
        how="left"
    )

    evaluated["test_hit"] = evaluated["test_hit"].fillna(0)

    # User-level HitRate@10
    user_hits = evaluated.groupby("user_id")["test_hit"].max()

    hit_rate = user_hits.mean()
    users_with_hits = (user_hits > 0).sum()

    # Exploration recommendation acceptance
    exploration_df = evaluated[
        evaluated["recommendation_type"] == "Exploration"
    ]

    if len(exploration_df) > 0:
        exploration_hit_rate = exploration_df["test_hit"].mean()
        exploration_hits = exploration_df["test_hit"].sum()
    else:
        exploration_hit_rate = np.nan
        exploration_hits = 0

    evaluation_results.append({
        "Exploration Rate": f"{int(rate * 100)}%",
        "HitRate@10": hit_rate,
        "Users with at least one hit": users_with_hits,
        "Exploration Recommendations": len(exploration_df),
        "Exploration Hits": int(exploration_hits),
        "Exploration Hit Rate": exploration_hit_rate
    })

evaluation_summary = pd.DataFrame(evaluation_results)

evaluation_summary

,Exploration Rate,HitRate@10,Users with at least one hit,Exploration Recommendations,Exploration Hits,Exploration Hit Rate
0,0%,0.892984,1260,0,0,NaN
1,10%,0.890149,1256,1411,277,0.196315
2,20%,0.883770,1247,2822,580,0.205528
3,30%,0.889440,1255,4233,864,0.204111


In [101]:
# Evaluate creator exposure and coverage

exposure_results = []

for rate, rec_df in controlled_recommendations.items():

    evaluated = rec_df.merge(
        creator_mapping[["video_id", "author_id", "creator_group"]],
        on="video_id",
        how="left"
    )

    print(f"\nExploration Rate: {int(rate * 100)}%")
    print("Missing creator groups:", evaluated["creator_group"].isna().sum())

    group_exposure = evaluated.groupby("creator_group").agg(
        recommendation_slots=("video_id", "count"),
        unique_creators=("author_id", "nunique")
    ).reset_index()

    group_exposure["exposure_share"] = (
        group_exposure["recommendation_slots"] / len(evaluated) * 100
    )

    for _, row in group_exposure.iterrows():

        population_count = (
            experiment_creator_stats[
                experiment_creator_stats["creator_group"] == row["creator_group"]
            ]["author_id"].nunique()
        )

        coverage = row["unique_creators"] / population_count * 100

        exposure_results.append({
            "Exploration Rate": f"{int(rate * 100)}%",
            "Creator Group": row["creator_group"],
            "Recommendation Slots": row["recommendation_slots"],
            "Exposure Share (%)": row["exposure_share"],
            "Unique Creators Reached": row["unique_creators"],
            "Creator Coverage (%)": coverage
        })

    print("Total unique creators recommended:",
          evaluated["author_id"].nunique())

exposure_summary = pd.DataFrame(exposure_results)

exposure_summary.round(2)


Exploration Rate: 0%
Missing creator groups: 0
Total unique creators recommended: 1069

Exploration Rate: 10%
Missing creator groups: 0
Total unique creators recommended: 1064

Exploration Rate: 20%
Missing creator groups: 0
Total unique creators recommended: 1060

Exploration Rate: 30%
Missing creator groups: 0
Total unique creators recommended: 1051


,Exploration Rate,Creator Group,Recommendation Slots,Exposure Share (%),Unique Creators Reached,Creator Coverage (%)
0,0%,Emerging,1422,10.08,240,47.24
1,0%,Established,12688,89.92,829,54.43
2,10%,Emerging,1429,10.13,237,46.65
3,10%,Established,12681,89.87,827,54.30
4,20%,Emerging,1452,10.29,241,47.44
5,20%,Established,12658,89.71,819,53.78
6,30%,Emerging,1447,10.26,236,46.46
7,30%,Established,12663,89.74,815,53.51


In [102]:
# Measure creator diversity per user

diversity_results = []

for rate, rec_df in controlled_recommendations.items():

    evaluated = rec_df.merge(
        creator_mapping[["video_id", "author_id"]],
        on="video_id",
        how="left"
    )

    creators_per_user = (
        evaluated.groupby("user_id")["author_id"]
        .nunique()
    )

    diversity_results.append({
        "Exploration Rate": f"{int(rate * 100)}%",
        "Mean Unique Creators": creators_per_user.mean(),
        "Median Unique Creators": creators_per_user.median(),
        "Minimum": creators_per_user.min(),
        "Maximum": creators_per_user.max()
    })

diversity_summary = pd.DataFrame(diversity_results)

diversity_summary.round(2)

,Exploration Rate,Mean Unique Creators,Median Unique Creators,Minimum,Maximum
0,0%,9.84,10.0,7,10
1,10%,9.84,10.0,7,10
2,20%,9.86,10.0,7,10
3,30%,9.86,10.0,7,10


In [103]:
# Build fairness-aware exploration candidates

fairness_strength = 0.08

fair_exploration = exploration_candidates.copy()

fair_exploration = fair_exploration.merge(
    creator_mapping[
        ["video_id", "author_id", "creator_group"]
    ],
    on="video_id",
    how="left"
)

fair_exploration["fairness_boost"] = np.where(
    fair_exploration["creator_group"] == "Emerging",
    fairness_strength,
    0
)

fair_exploration["combined_exploration_score"] = (
    fair_exploration["smoothed_interest"]
    + fair_exploration["fairness_boost"]
)

# Sort by user and combined score
fair_exploration = fair_exploration.sort_values(
    ["user_id", "combined_exploration_score"],
    ascending=[True, False]
)

print("Fairness-aware candidates:", len(fair_exploration))
print("Users:", fair_exploration["user_id"].nunique())

print("\nCreator group distribution:")
print(fair_exploration["creator_group"].value_counts())

print("\nDuplicate user-video pairs:",
      fair_exploration.duplicated(
          subset=["user_id", "video_id"]
      ).sum())

fair_exploration.head()

Fairness-aware candidates: 47803
Users: 1411

Creator group distribution:
creator_group
Established    42912
Emerging        4891
Name: count, dtype: int64

Duplicate user-video pairs: 0


,user_id,video_id,smoothed_interest,author_id,creator_group,fairness_boost,combined_exploration_score
26,14,8624,1.119274,2193,Established,0.00,1.119274
2,14,835,1.104619,1481,Established,0.00,1.104619
7,14,2365,1.104619,7128,Established,0.00,1.104619
5,14,2102,1.103374,6340,Established,0.00,1.103374
28,14,9986,1.013739,3711,Emerging,0.08,1.093739


In [104]:
# Select top 3 fairness-aware exploration videos per user

fair_exploration_recommendations = (
    fair_exploration
    .sort_values(
        ["user_id", "combined_exploration_score"],
        ascending=[True, False]
    )
    .groupby("user_id")
    .head(3)
    .copy()
)

fair_exploration_recommendations["recommendation_type"] = "Fair Exploration"

print("Total recommendations:",
      len(fair_exploration_recommendations))

print("Users:",
      fair_exploration_recommendations["user_id"].nunique())

print("\nRecommendations per user:")
print(
    fair_exploration_recommendations
    .groupby("user_id")
    .size()
    .value_counts()
)

print("\nDuplicate user-video pairs:",
      fair_exploration_recommendations.duplicated(
          subset=["user_id", "video_id"]
      ).sum())

print("\nCreator group distribution:")
print(
    fair_exploration_recommendations["creator_group"]
    .value_counts()
)

fair_exploration_recommendations.head(10)

Total recommendations: 4233
Users: 1411

Recommendations per user:
3    1411
Name: count, dtype: int64

Duplicate user-video pairs: 0

Creator group distribution:
creator_group
Established    3405
Emerging        828
Name: count, dtype: int64


,user_id,video_id,smoothed_interest,author_id,creator_group,fairness_boost,combined_exploration_score,recommendation_type
26,14,8624,1.119274,2193,Established,0.00,1.119274,Fair Exploration
2,14,835,1.104619,1481,Established,0.00,1.104619,Fair Exploration
7,14,2365,1.104619,7128,Established,0.00,1.104619,Fair Exploration
43,19,4050,0.968924,2056,Established,0.00,0.968924,Fair Exploration
51,19,8361,0.968924,3281,Established,0.00,0.968924,Fair Exploration
38,19,2588,0.886046,2390,Emerging,0.08,0.966046,Fair Exploration
92,21,10405,1.137878,5680,Established,0.00,1.137878,Fair Exploration
87,21,9109,1.100608,2682,Established,0.00,1.100608,Fair Exploration
73,21,3102,1.086823,1474,Established,0.00,1.086823,Fair Exploration
99,23,3247,1.383505,5588,Established,0.00,1.383505,Fair Exploration


In [105]:
# Evaluate fairness-aware exploration relevance

fair_evaluated = fair_exploration_recommendations.merge(
    test_pairs,
    on=["user_id", "video_id"],
    how="left"
)

fair_evaluated["test_hit"] = (
    fair_evaluated["test_hit"].fillna(0)
)

# Overall exploration relevance
total_hits = fair_evaluated["test_hit"].sum()
total_recommendations = len(fair_evaluated)

print("Total recommendations:", total_recommendations)
print("Total test hits:", int(total_hits))

print(
    "Overall exploration Hit Rate:",
    round(fair_evaluated["test_hit"].mean() * 100, 2), "%"
)

# Relevance by creator group
fair_group_evaluation = fair_evaluated.groupby(
    "creator_group"
).agg(
    recommendations=("video_id", "count"),
    test_hits=("test_hit", "sum")
).reset_index()

fair_group_evaluation["hit_rate (%)"] = (
    fair_group_evaluation["test_hits"]
    / fair_group_evaluation["recommendations"] * 100
)

fair_group_evaluation.round(2)

Total recommendations: 4233
Total test hits: 789
Overall exploration Hit Rate: 18.64 %


,creator_group,recommendations,test_hits,hit_rate (%)
0,Emerging,828,67.0,8.09
1,Established,3405,722.0,21.20


In [106]:
# Build final fairness-aware Top-10 recommendations

fairness_aware_recommendations = []

for user_id in train_data["user_id"].unique():

    # Get the 3 fairness-aware exploration videos
    user_fair = fair_exploration_recommendations[
        fair_exploration_recommendations["user_id"] == user_id
    ]

    selected_ids = set(user_fair["video_id"])

    # Get 7 normal videos, excluding exploration selections
    user_normal = normal_candidates[
        (normal_candidates["user_id"] == user_id) &
        (~normal_candidates["video_id"].isin(selected_ids))
    ].sort_values(
        "interest_score",
        ascending=False
    ).head(7)

    # Add normal recommendations
    for _, row in user_normal.iterrows():
        fairness_aware_recommendations.append({
            "user_id": user_id,
            "video_id": row["video_id"],
            "recommendation_type": "Normal"
        })

    # Add fairness-aware exploration recommendations
    for _, row in user_fair.iterrows():
        fairness_aware_recommendations.append({
            "user_id": user_id,
            "video_id": row["video_id"],
            "recommendation_type": "Fair Exploration"
        })

fairness_aware_recommendations = pd.DataFrame(
    fairness_aware_recommendations
)

# Structural validation
print("Total recommendations:",
      len(fairness_aware_recommendations))

print("Users:",
      fairness_aware_recommendations["user_id"].nunique())

print("\nRecommendation types:")
print(fairness_aware_recommendations["recommendation_type"].value_counts())

print("\nDuplicate user-video pairs:",
      fairness_aware_recommendations.duplicated(
          subset=["user_id", "video_id"]
      ).sum())

unique_counts = (
    fairness_aware_recommendations
    .groupby("user_id")["video_id"]
    .nunique()
)

print("\nMinimum unique videos per user:", unique_counts.min())
print("Maximum unique videos per user:", unique_counts.max())

Total recommendations: 14110
Users: 1411

Recommendation types:
recommendation_type
Normal              9877
Fair Exploration    4233
Name: count, dtype: int64

Duplicate user-video pairs: 0

Minimum unique videos per user: 10
Maximum unique videos per user: 10


In [107]:
# Compare recommendation quality across strategies

strategies = {
    "Baseline (0%)": controlled_recommendations[0.0],
    "Ordinary Exploration (30%)": controlled_recommendations[0.3],
    "Fairness-Aware (30%)": fairness_aware_recommendations
}

final_quality_results = []

for strategy_name, rec_df in strategies.items():

    evaluated = rec_df.merge(
        test_pairs,
        on=["user_id", "video_id"],
        how="left"
    )

    evaluated["test_hit"] = evaluated["test_hit"].fillna(0)

    # User-level HitRate@10
    user_hits = evaluated.groupby("user_id")["test_hit"].max()

    hit_rate = user_hits.mean()
    users_with_hits = (user_hits > 0).sum()

    # Slot-level hit rate
    slot_hit_rate = evaluated["test_hit"].mean()

    final_quality_results.append({
        "Strategy": strategy_name,
        "HitRate@10 (%)": hit_rate * 100,
        "Users with at least one hit": users_with_hits,
        "Recommendation Slot Hit Rate (%)": slot_hit_rate * 100
    })

final_quality_summary = pd.DataFrame(final_quality_results)

final_quality_summary.round(2)

,Strategy,HitRate@10 (%),Users with at least one hit,Recommendation Slot Hit Rate (%)
0,Baseline (0%),89.30,1260,22.71
1,Ordinary Exploration (30%),88.94,1255,21.79
2,Fairness-Aware (30%),88.09,1243,21.35


In [108]:
# Compare creator exposure across strategies

exposure_strategies = {
    "Baseline (0%)": controlled_recommendations[0.0],
    "Ordinary Exploration (30%)": controlled_recommendations[0.3],
    "Fairness-Aware (30%)": fairness_aware_recommendations
}

final_exposure_results = []

for strategy_name, rec_df in exposure_strategies.items():

    evaluated = rec_df.merge(
        creator_mapping[
            ["video_id", "author_id", "creator_group"]
        ],
        on="video_id",
        how="left"
    )

    print(strategy_name, "- Missing creator groups:",
          evaluated["creator_group"].isna().sum())

    group_stats = evaluated.groupby("creator_group").agg(
        recommendation_slots=("video_id", "count"),
        unique_creators=("author_id", "nunique")
    )

    for group in ["Emerging", "Established"]:

        slots = group_stats.loc[group, "recommendation_slots"]
        unique_creators = group_stats.loc[group, "unique_creators"]

        population_count = experiment_creator_stats[
            experiment_creator_stats["creator_group"] == group
        ]["author_id"].nunique()

        final_exposure_results.append({
            "Strategy": strategy_name,
            "Creator Group": group,
            "Exposure (%)": slots / len(evaluated) * 100,
            "Recommendation Slots": slots,
            "Unique Creators Reached": unique_creators,
            "Creator Coverage (%)": (
                unique_creators / population_count * 100
            )
        })

final_exposure_summary = pd.DataFrame(final_exposure_results)

final_exposure_summary.round(2)

Baseline (0%) - Missing creator groups: 0
Ordinary Exploration (30%) - Missing creator groups: 0
Fairness-Aware (30%) - Missing creator groups: 0


,Strategy,Creator Group,Exposure (%),Recommendation Slots,Unique Creators Reached,Creator Coverage (%)
0,Baseline (0%),Emerging,10.08,1422,240,47.24
1,Baseline (0%),Established,89.92,12688,829,54.43
2,Ordinary Exploration (30%),Emerging,10.26,1447,236,46.46
3,Ordinary Exploration (30%),Established,89.74,12663,815,53.51
4,Fairness-Aware (30%),Emerging,12.67,1788,244,48.03
5,Fairness-Aware (30%),Established,87.33,12322,815,53.51


In [110]:
# Recalculate Emerging creator exposure gap

emerging_population_share = (
    experiment_creator_stats["creator_group"]
    .eq("Emerging")
    .mean() * 100
)

emerging_results = final_exposure_summary[
    final_exposure_summary["Creator Group"] == "Emerging"
].copy()

emerging_results["Population Share (%)"] = emerging_population_share

emerging_results["Exposure Gap (pp)"] = (
    emerging_results["Exposure (%)"]
    - emerging_population_share
)

baseline_exposure = emerging_results.iloc[0]["Exposure (%)"]

emerging_results["Exposure Improvement vs Baseline (pp)"] = (
    emerging_results["Exposure (%)"] - baseline_exposure
)

emerging_gap_summary = emerging_results[
    [
        "Strategy",
        "Exposure (%)",
        "Population Share (%)",
        "Exposure Gap (pp)",
        "Exposure Improvement vs Baseline (pp)"
    ]
]

print("Emerging creator population share:",
      round(emerging_population_share, 2), "%")

display(emerging_gap_summary.round(2))

Emerging creator population share: 25.01 %


,Strategy,Exposure (%),Population Share (%),Exposure Gap (pp),Exposure Improvement vs Baseline (pp)
0,Baseline (0%),10.08,25.01,-14.93,0.00
2,Ordinary Exploration (30%),10.26,25.01,-14.76,0.18
4,Fairness-Aware (30%),12.67,25.01,-12.34,2.59


# Notebook 06: Evaluation — Final Findings

## 1. Experimental Setup

The evaluation compared three recommendation strategies:

- Baseline: 0% exploration.
- Ordinary exploration: 30% exploration.
- Fairness-aware exploration: 30% exploration with an additional fairness boost of 0.08 for Emerging creators.

Each strategy generated 10 unique recommendations for each of 1,411 users, resulting in 14,110 recommendations per strategy.

## 2. Recommendation Quality

| Strategy | HitRate@10 |
|---|---:|
| Baseline | 89.30% |
| Ordinary Exploration | 88.94% |
| Fairness-Aware | 88.09% |

The fairness-aware strategy achieved an absolute HitRate@10 reduction of 1.21 percentage points compared with baseline.

## 3. Creator Exposure

| Strategy | Emerging Exposure |
|---|---:|
| Baseline | 10.08% |
| Ordinary Exploration | 10.26% |
| Fairness-Aware | 12.67% |

Emerging creators represent 25.01% of the creator population.

The fairness-aware strategy improved Emerging creator exposure by 2.59 percentage points compared with baseline, while ordinary exploration improved it by only 0.18 percentage points.

## 4. Exposure Gap

The Emerging creator exposure gap decreased from -14.93 percentage points under baseline to -12.34 percentage points under fairness-aware exploration.

Although the fairness-aware strategy improved exposure, it did not achieve population-proportional representation.

## 5. Research Findings

The experiments demonstrate that ordinary exploration alone does not substantially address creator exposure imbalance.

Introducing a fairness boost into exploration ranking increases exposure for Emerging creators while retaining a relatively high HitRate@10.

The results highlight a measurable trade-off between recommendation relevance and creator exposure fairness.

## 6. Limitations

- Evaluation uses an offline temporal holdout rather than live user interactions.
- Historical interactions may reflect existing platform exposure biases.
- A held-out interaction hit does not necessarily represent user satisfaction.
- The fairness boost was evaluated at a particular setting and requires sensitivity analysis.
- Creator population representation is used as a reference point, but equal exposure is not necessarily appropriate for every platform or recommendation context.
- Results are based on one dataset and one temporal split.

## 7. Conclusion

The Spotlight framework demonstrates that interest-constrained, fairness-aware exploration can improve exposure opportunities for Emerging creators without completely sacrificing recommendation relevance.

The remaining exposure gap indicates that further improvements to fairness-aware ranking and exploration allocation are necessary.